<a href="https://colab.research.google.com/github/one-2730/ESSA-25-1/blob/Assignment/ESAA_OB_0411_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#09 추천시스템

**추천시스템**

- 활용 도메인: 전자상거래(아마존 등), 콘텐츠 포털(유튜브, 애플 뮤직 등)

- 필요성: 다품목화된 온라인 스토어에서 시간 제약으로 인해 소비자가 결정에 어려움을 겪으면 매출 감소로 이어질 수 있음

- 필요한 데이터
  - 구매 이력
  - 장바구니, 조회 이력
  - 제품 평가 이력
  - 스스로 작성한 선호
  - 클릭 이력

- 유형
  - 콘텐츠 기반 필터링(Content based filtering)
    - 사용자가 특정한 아이템을 선호하는 경우, 그 아이템과 유사한 다른 아이템 추천
  - 협업 필터링(Collaborative filtering)
    - 사용자 행동 양식을 기반으로 추천 수행
    - 사용자-아이템 평점 행렬 데이터에만 의지하여 추천 수행
    - 유형
      - 최근접 이웃
        - 사용자 기반(User-User)
        - 아이템 기반(Item-Item)
      - 잠재 요인(Latent factor): 행렬 분해(Matrix Factorization) 기법 활용

In [3]:
# 잠재 요인 협업 필터링 구현
# SDG 기반 행렬 분해

import numpy as np

R = np.array([[4, np.nan, np.nan, 2, np.nan],
              [np.nan, 5, np.nan, 3, 1],
              [np.nan, np.nan, 3, 4, 4],
              [5, 2, 1, 2, np.nan]])
num_users, num_items = R.shape
k=3

#P와 Q 행렬 크기를 지정하고 정규분포에서 랜덤 지정
np.random.seed(1)
P = np.random.normal(scale=1./k, size=(num_users, k))
Q = np.random.normal(scale=1./k, size=(num_items, k))

from sklearn.metrics import mean_squared_error

def get_rmse(R, P, Q, non_zeros):
  error = 0
  full_pred_matrix = np.dot(P, Q.T)

  x_non_zero_ind = [non_zero[0] for non_zero in non_zeros]
  y_non_zero_ind = [non_zero[1] for non_zero in non_zeros]
  R_non_zeros = R[x_non_zero_ind, y_non_zero_ind]
  full_pred_matrix_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind]
  mse = mean_squared_error(R_non_zeros, full_pred_matrix_non_zeros)
  rmse = np.sqrt(mse)

  return rmse

In [4]:
non_zeros = [(i, j, R[i, j]) for i in range(num_users) for j in range(num_items) if R[i, j]>0]

steps = 1000
learning_rate = 0.1
r_lambda = 0.01

for step in range(steps):
  for i, j, r in non_zeros:
    eij = r - np.dot(P[i, :], Q[j, :].T)
    P[i, :] = P[i, :]+learning_rate*(eij*Q[j, :]-r_lambda*P[i, :])
    Q[j, :] = Q[j, :]+learning_rate*(eij*P[i, :]-r_lambda*Q[j, :])

    rmse = get_rmse(R, P, Q, non_zeros)
    if (step % 50) == 0:
      print("### iteration step : ", step, "rmse : ", rmse)

### iteration step :  0 rmse :  3.25187845360079
### iteration step :  0 rmse :  3.2444052260039506
### iteration step :  0 rmse :  3.167084216936956
### iteration step :  0 rmse :  3.1495983559019014
### iteration step :  0 rmse :  3.1522336245710996
### iteration step :  0 rmse :  3.1351500912707544
### iteration step :  0 rmse :  3.0969531517038758
### iteration step :  0 rmse :  3.0620074299003566
### iteration step :  0 rmse :  3.034770934658341
### iteration step :  0 rmse :  3.017899599963068
### iteration step :  0 rmse :  3.016150964663611
### iteration step :  0 rmse :  2.9960320460843546
### iteration step :  50 rmse :  0.07395440138778553
### iteration step :  50 rmse :  0.06293874014868941
### iteration step :  50 rmse :  0.06276280567104289
### iteration step :  50 rmse :  0.06202583363084654
### iteration step :  50 rmse :  0.06217862283901784
### iteration step :  50 rmse :  0.061965052827622095
### iteration step :  50 rmse :  0.06205093549860854
### iteration step :  

In [5]:
pred_matrix = np.dot(P, Q.T)
print('예측 행렬:\n', np.round(pred_matrix, 3))

예측 행렬:
 [[3.991 1.708 1.128 1.999 1.662]
 [5.813 4.977 0.597 2.986 1.002]
 [6.696 1.761 2.987 3.982 3.991]
 [4.973 1.991 1.    2.001 1.727]]
